# NB‑01 — Anchor Check (Retriever Goldens)

Companion to `doc/design/01_lld_tests.md` and `doc/design/01_authoring_recipe.md`.

This notebook validates `eval/goldens/retriever_goldens.json` against the Mod‑01 contract:
- Exact 8‑field schema
- Category enum and minimums
- ID format + uniqueness
- 100% groundedness (ideal_context + must_contain in corpus)
- source/path truthfulness (derived from manifest)
- No secrets
- Counts in target ranges

All checks are offline and deterministic. On any failure, this notebook raises an exception so CI can fail.

In [7]:
import json
import os
import re
import sys
from pathlib import Path
from collections import Counter

from dotenv import load_dotenv

# --- Repo root resolution ---------------------------------------------
# Makes the relative "data/..." and "eval/..." paths work no matter
# where Jupyter was launched from.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

assert (ROOT / "pyproject.toml").exists(), (
    f"pyproject.toml not found in any parent directory — stopped at {ROOT}"
)

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
load_dotenv(ROOT / ".env", override=True)

print("repo root:", ROOT)

# --- Paths --------------------------------------------------------------
GOLD = ROOT / "eval" / "goldens" / "retriever_goldens.json"
DATA = ROOT / "data" / "docs"

# --- Corpus manifest (S1..S5) -------------------------------------------
SRC = {
    "S1": "s1_early_steps_bundle.md",
    "S2": "s2_frameworks_bundle.md",
    "S3": "s3_lifecycle_bundle.md",
    "S4": "s4_qa_deep_dives.md",
    "S5": "s5_docs_bundle.md",
}

# --- Schema contract ------------------------------------------------------
KEYS = {
    "id",
    "category",
    "query",
    "ideal_answer",
    "ideal_context",
    "must_contain",
    "source",
    "path",
}

CATS = {
    "cite",
    "conflict",
    "misroute",
    "abstain",
    "degrade",
    "multi-source",
    "basic",
}

# ID regex: S[1-5]-Q<seq> or MS-Q<seq>
IDRE = re.compile(r"^(S[1-5]-Q\d+|MS-Q\d+)$")

# Secret patterns (simplified version of recipe)
SEC = re.compile(
    r"(sk-[A-Za-z0-9]{20,}"
    r"|api[_-]?key\s*[:=]\s*['\"][A-Za-z0-9_\-]{16,}"
    r"|AIza[A-Za-z0-9_\-]{20,}"
    r"|gsk_[A-Za-z0-9]{20,})",
    re.I
)

repo root: /home/dipak/agentic/step9_llmops


In [9]:
import os
from pathlib import Path

print("DATA resolves to:", Path(DATA).resolve())
print("Exists?", Path(DATA).exists())
print("Contents:", list(Path(DATA).glob("**/*")) if Path(DATA).exists() else "N/A")

DATA resolves to: /home/dipak/agentic/step9_llmops/data/docs
Exists? True
Contents: [PosixPath('/home/dipak/agentic/step9_llmops/data/docs/s1_early_steps_bundle.md'), PosixPath('/home/dipak/agentic/step9_llmops/data/docs/s2_frameworks_bundle.md'), PosixPath('/home/dipak/agentic/step9_llmops/data/docs/s3_lifecycle_bundle.md'), PosixPath('/home/dipak/agentic/step9_llmops/data/docs/s5_docs_bundle.md'), PosixPath('/home/dipak/agentic/step9_llmops/data/docs/s4_qa_deep_dives.md')]


In [10]:
bund = {}
missing = []
for src, filename in SRC.items():
    path = Path(DATA) / filename
    if not path.exists():
        missing.append(str(path))
        continue
    bund[src] = path.read_text(encoding="utf-8")
    print(f"Loaded {src}: {len(bund[src])} chars from {path}")

if missing:
    print("Missing files:", missing)

Loaded S1: 60788 chars from /home/dipak/agentic/step9_llmops/data/docs/s1_early_steps_bundle.md
Loaded S2: 31136 chars from /home/dipak/agentic/step9_llmops/data/docs/s2_frameworks_bundle.md
Loaded S3: 16893 chars from /home/dipak/agentic/step9_llmops/data/docs/s3_lifecycle_bundle.md
Loaded S4: 32519 chars from /home/dipak/agentic/step9_llmops/data/docs/s4_qa_deep_dives.md
Loaded S5: 33815 chars from /home/dipak/agentic/step9_llmops/data/docs/s5_docs_bundle.md


In [11]:
# Load retriever goldens
gold_path = Path(GOLD)
rows = json.loads(gold_path.read_text(encoding="utf-8"))

assert isinstance(rows, list), "Goldens must be a JSON list"
print(f"Loaded {len(rows)} golden rows from {GOLD}")

Loaded 139 golden rows from /home/dipak/agentic/step9_llmops/eval/goldens/retriever_goldens.json


In [23]:
print(len(rows))

139


In [12]:
# T-01-1: Counts in range (retriever goldens: 120–160)
n = len(rows)
assert 120 <= n <= 160, f"T-01-1 FAIL: count={n}, expected 120–160"
print(f"T-01-1 PASS: counts in range (n={n})")

T-01-1 PASS: counts in range (n=139)


In [13]:
# T-01-2: Schema exactness, category enum, must_contain shape
schema_failures = []

for r in rows:
    rid = r.get("id", "?")

    # Exact keys
    if set(r.keys()) != KEYS:
        schema_failures.append((rid, "keys", sorted(set(r.keys()) ^ KEYS)))

    # Category enum
    if r["category"] not in CATS:
        schema_failures.append((rid, "category", r["category"]))

    # must_contain shape: non-empty list of non-empty strings
    mc = r.get("must_contain")
    if not isinstance(mc, list) or len(mc) == 0:
        schema_failures.append((rid, "must_contain_type", "not a non-empty list"))
    elif not all(isinstance(x, str) and x for x in mc):
        schema_failures.append((rid, "must_contain_type", "contains non-string or empty"))

assert len(schema_failures) == 0, f"T-01-2 FAIL: {schema_failures}"
print("T-01-2 PASS: schema, category enum, must_contain shape")

T-01-2 PASS: schema, category enum, must_contain shape


In [14]:
# T-01-4: ID regex + uniqueness (ids and queries)
id_failures = []

ids = [r["id"] for r in rows]
qs = [r["query"] for r in rows]

# ID regex
for r in rows:
    if not IDRE.match(r["id"]):
        id_failures.append((r["id"], "id_regex"))

# Duplicates
if len(ids) != len(set(ids)):
    id_failures.append(("ids", "dup"))
if len(qs) != len(set(qs)):
    id_failures.append(("queries", "dup"))

assert len(id_failures) == 0, f"T-01-4 FAIL: {id_failures}"
print("T-01-4 PASS: ID regex + uniqueness")

T-01-4 PASS: ID regex + uniqueness


In [15]:
# Helper: derive expected path from manifest (T-01-5)
def expect_path(srcs):
    return " + ".join("data/docs/" + SRC[s] for s in srcs)

In [16]:
# T-01-3 & T-01-5: Groundedness + source/path truth
grounding_failures = []

for r in rows:
    rid = r["id"]
    srcs = r["source"].split("+")

    # Source validity
    if any(s not in SRC for s in srcs):
        grounding_failures.append((rid, "source", srcs))
        continue

    # must_contain union-grounding
    miss = [
        f for f in r["must_contain"]
        if not any(f in bund[s] for s in srcs)
    ]
    if miss:
        grounding_failures.append((rid, "must_contain_grounding", miss))

    # ideal_context contiguity
    if not any(r["ideal_context"] in bund[s] for s in srcs):
        grounding_failures.append((rid, "context_not_contiguous"))

    # path truthfulness
    expected_path = expect_path(srcs)
    if r["path"] != expected_path:
        grounding_failures.append((rid, "path", {"got": r["path"], "expected": expected_path}))

assert len(grounding_failures) == 0, f"T-01-3/T-01-5 FAIL: {grounding_failures}"
print("T-01-3 & T-01-5 PASS: groundedness + source/path truth")

T-01-3 & T-01-5 PASS: groundedness + source/path truth


In [17]:
# T-01-7: No secrets
secret_failures = []

for r in rows:
    rid = r["id"]
    text = (
        r["ideal_answer"]
        + r["ideal_context"]
        + " ".join(r["must_contain"])
    )
    if SEC.search(text):
        secret_failures.append((rid, "secret"))

assert len(secret_failures) == 0, f"T-01-7 FAIL: {secret_failures}"
print("T-01-7 PASS: no secrets")

T-01-7 PASS: no secrets


In [18]:
# T-01-6: Category minimums + report
cc = Counter(r["category"] for r in rows)
print("Category counts:", dict(cc))

edge_mins = {
    "misroute": 5,
    "conflict": 5,
    "abstain": 5,
    "degrade": 5,
}

cat_failures = []
for cat, min_count in edge_mins.items():
    actual = cc.get(cat, 0)
    if actual < min_count:
        cat_failures.append((cat, actual, min_count))

assert len(cat_failures) == 0, f"T-01-6 FAIL (edge category minimums): {cat_failures}"
print(
    "T-01-6 PASS: edge category minimums",
    ", ".join(f"{k}={cc.get(k,0)}" for k in edge_mins)
)

Category counts: {'multi-source': 1, 'conflict': 5, 'degrade': 5, 'basic': 23, 'cite': 95, 'abstain': 5, 'misroute': 5}
T-01-6 PASS: edge category minimums misroute=5, conflict=5, abstain=5, degrade=5


### T‑01‑9: Answerable-from-corpus-only (spot-check)

Full automation is impossible here. This cell:
- Runs a simple heuristic (key query terms appear in the corresponding source bundle).
- Prints a sample of queries per source for manual reviewer.

Treat heuristic failures as warnings; manual review is the real gate.

In [19]:
# T-01-9: Answerable-from-corpus-only (heuristic + spot-check sample)
import random

random.seed(0)

# Simple heuristic: at least one non-trivial word from query appears in source bundle(s)
heuristic_failures = []

for r in rows:
    rid = r["id"]
    srcs = r["source"].split("+")
    query = r["query"]

    # Very simple tokenization
    tokens = [t for t in query.lower().split() if len(t) > 4]
    if not tokens:
        continue

    found = False
    for s in srcs:
        txt = bund[s].lower()
        if any(t in txt for t in tokens):
            found = True
            break

    if not found:
        heuristic_failures.append((rid, query))

if heuristic_failures:
    print("T-01-9 HEURISTIC WARNINGS (review manually):")
    for rid, q in heuristic_failures[:20]:
        print(f"  {rid}: {q[:80]}...")
else:
    print("T-01-9 heuristic: no warnings")

# Spot-check sample: print 5 random queries per source
print("\nT-01-9 spot-check sample (for manual review):")
by_source = {}
for r in rows:
    srcs = r["source"].split("+")
    for s in srcs:
        by_source.setdefault(s, []).append(r)

for s in sorted(by_source):
    sample = random.sample(by_source[s], min(5, len(by_source[s])))
    print(f"\n{s} sample:")
    for r in sample:
        print(f"  {r['id']}: {r['query'][:80]}")

T-01-9 heuristic: no warnings

T-01-9 spot-check sample (for manual review):

S1 sample:
  S1-Q023: Which metric maps to which retrieval failure in step 2's symptom table?
  S1-Q025: What fields does each golden.jsonl entry carry in step 2?
  S1-Q001: What chunking strategy does step 1 use, and what are the chunk size and overlap?
  S1-Q015: How does the shared prompt registry treat the steps that follow step 1?
  S1-Q031: What eval scores does the RAG_ANSWER V1 prompt record?

S2 sample:
  S2-Q014: When does step 5 suggest adding a refuse node?
  S2-Q011: Which embedding model does step 3 record, and why that choice?
  S2-Q024: How does step 5 handle a low retrieval grade at run time?
  S2-Q025: Where does LangGraph enter the pipeline series?
  S2-Q008: What is the core design insight about LangChain and the PromptRegistry?

S3 sample:
  S3-Q014: With a $5 daily budget and 2000 queries/day, can you afford to switch to the bet
  S3-Q010: What was the measured outcome of the V3 prompt a

In [20]:
# Cell 12 — Summary & exit status
print("\n" + "="*60)
print("NB-01 ANCHOR CHECK SUMMARY")
print("="*60)

# All checks passed if we reached here
print("T-01-1  PASS: counts in range")
print("T-01-2  PASS: schema, category enum, must_contain shape")
print("T-01-3  PASS: groundedness (ideal_context + must_contain)")
print("T-01-4  PASS: ID regex + uniqueness")
print("T-01-5  PASS: source/path truth")
print("T-01-6  PASS: category minimums")
print("T-01-7  PASS: no secrets")
print("T-01-9  INFO: heuristic + spot-check printed above")

print("\nTOTAL", len(rows), "| FAILURES: NONE")
print("✅ NB-01 anchor check: ALL TESTS PASSED")


NB-01 ANCHOR CHECK SUMMARY
T-01-1  PASS: counts in range
T-01-2  PASS: schema, category enum, must_contain shape
T-01-3  PASS: groundedness (ideal_context + must_contain)
T-01-4  PASS: ID regex + uniqueness
T-01-5  PASS: source/path truth
T-01-6  PASS: category minimums
T-01-7  PASS: no secrets
T-01-9  INFO: heuristic + spot-check printed above

TOTAL 139 | FAILURES: NONE
✅ NB-01 anchor check: ALL TESTS PASSED


### T‑01‑8: step4 frozen (optional / process gate)

If you maintain a snapshot of step4 golden hashes, implement the comparison here.
For now, this is a placeholder for that process.

In [21]:
# T-01-8: step4 frozen (placeholder)
# TODO: compare current step4 golden hashes to stored snapshot.
# For now, just print a reminder.
print("T-01-8: step4 frozen check — implement hash comparison against stored snapshot.")

T-01-8: step4 frozen check — implement hash comparison against stored snapshot.


Generted code of golden json

In [24]:
# ============================================================
# Golden dataset authoring cell
# Implements: doc/design/01_authoring_recipe.md, sections 1-5
# ============================================================
import json
import re
from pathlib import Path

# --- 1. Path-derivation helper (T-01-5 contract) --------------------------
def expect_path(srcs):
    """Given a list of source keys (e.g. ['S1'] or ['S1','S2']),
    derive the canonical 'path' string from the manifest — never type it."""
    return " + ".join("data/docs/" + SRC[s] for s in srcs)


# --- 3. Anchor extraction helper — byte-exact, fails loudly ----------------
def A(lines, full, nums, marker):
    """
    lines  : bund[src].splitlines(keepends=True)
    full   : bund[src] (the whole file text)
    nums   : 1-based line numbers to extract (must be contiguous)
    marker : a substring that MUST appear in the extracted excerpt

    Returns the byte-exact excerpt (rstripped of trailing newline).
    Raises AssertionError immediately on:
      - marker not found in the excerpt (likely off-by-one line numbers)
      - excerpt not found verbatim in the full text (non-contiguous, e.g.
        a skipped blank line between anchors)
    """
    txt = "".join(lines[i - 1] for i in nums).rstrip("\n")
    assert marker in txt, f"MARKER FAIL on lines {nums}: {marker!r} -> {txt[:70]!r}"
    assert txt in full, f"NOT CONTIGUOUS lines {nums}"
    return txt


# --- 2. Anchor discovery helper (optional convenience) ---------------------
def find_anchor(src, token):
    """Scan bund[src] for lines containing `token`; returns [(line_no, line_text), ...]
    for manual inspection before writing A(...) calls."""
    lines = bund[src].splitlines(keepends=True)
    return [
        (i, lines[i - 1].rstrip("\n"))
        for i in range(1, len(lines) + 1)
        if token in lines[i - 1]
    ]


# --- 4. Row template ---------------------------------------------------
# Fill this in per new row. Example (adapt id/category/query/etc.):
#
# lines_s1 = bund["S1"].splitlines(keepends=True)
# ctx = A(lines_s1, bund["S1"], [310, 311, 312, 313, 314], "ask(question, top_k=3)")
#
# new_row = {
#     "id": "S1-Q014",
#     "category": "basic",
#     "query": "What happens in the ask() flow of step 1's pipeline?",
#     "ideal_answer": "ask(question, top_k=3) embeds the question, retrieves the "
#                      "top-k chunks, generates an answer with that context, and "
#                      "returns the answer plus citations.",
#     "ideal_context": ctx,
#     "must_contain": [
#         "ask(question, top_k=3)",
#         "retrieve top-k chunks",
#         "return answer + citations",
#     ],
#     "source": "S1",
#     "path": expect_path(["S1"]),
# }

NEW = [
    # new_row,   # <- append each authored row dict here
]


# --- 5. Batch append + deterministic sort + atomic write -------------------
def seq(rid: str) -> int:
    return int(re.search(r"Q(\d+)", rid).group(1))

def sort_key(r):
    prefix = r["id"][:2]
    order = 0 if r["id"].startswith("MS") else {"S1": 1, "S2": 2, "S3": 3, "S4": 4, "S5": 5}[prefix]
    return (order, seq(r["id"]))

def append_golden_rows(new_rows, gold_path=GOLD):
    """Idempotent: reloads committed rows, appends NEW, sorts, writes atomically."""
    gold_path = Path(gold_path)
    rows = json.loads(gold_path.read_text(encoding="utf-8"))

    existing_ids = {r["id"] for r in rows}
    dupes = [r["id"] for r in new_rows if r["id"] in existing_ids]
    assert not dupes, f"Refusing to append: id(s) already present {dupes}"

    rows.extend(new_rows)
    rows.sort(key=sort_key)

    tmp_path = gold_path.with_suffix(".json.tmp")
    tmp_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp_path.replace(gold_path)  # atomic on POSIX

    print(f"Appended {len(new_rows)} row(s). File now has {len(rows)} rows.")
    return rows


if NEW:
    rows = append_golden_rows(NEW)
else:
    print("NEW is empty — fill in row(s) above, then re-run this cell.")

NEW is empty — fill in row(s) above, then re-run this cell.
